# Pairs trading and mean reversion

Four screamer operators that handle cointegrated pairs: `RollingBeta` estimates the hedge ratio, `RollingSpread` extracts the beta-adjusted residual, `RollingAlpha` tracks the intercept, and `RollingOU` fits an Ornstein-Uhlenbeck model to quantify how fast the spread reverts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import RollingBeta, RollingSpread, RollingAlpha, RollingOU

rng = np.random.default_rng(0)

def show(ax, title, xlabel="sample"):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8)
    plt.tight_layout()

# Synthetic cointegrated pair
# A common random-walk factor shared by both assets
N = 500
beta_true = 1.4
a0 = 10.0

factor = np.cumsum(rng.standard_normal(N) * 0.5)
noise_a = rng.standard_normal(N) * 0.3
noise_b = rng.standard_normal(N) * 0.3

# Asset A: a0 + beta_true * factor + small idiosyncratic noise
# Asset B: factor + small idiosyncratic noise
# The true spread a - beta_true*b = a0 + noise_a - beta_true*noise_b, which is stationary
b = factor + noise_b
a = a0 + beta_true * factor + noise_a

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(a, lw=1.2, color="steelblue", label="asset A")
ax.plot(b, lw=1.2, color="darkorange", label="asset B (factor)")
show(ax, "Synthetic cointegrated pair")

## Hedge ratio with RollingBeta

`RollingBeta(window_size)(x, y)` computes the rolling regression slope of `x` on `y` over a trailing window: `beta = cov(x, y) / var(y)`. The first argument is the dependent variable (here asset A) and the second is the independent variable (asset B). As the window fills, the estimated ratio converges toward the true value of 1.4 used to construct the pair.

In [ ]:
W = 80
beta_est = RollingBeta(window_size=W)(a, b)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axhline(beta_true, color="0.35", ls="--", lw=1.2, label=f"true beta = {beta_true}")
ax.plot(beta_est, lw=1.8, color="steelblue", label=f"RollingBeta(window_size={W})")
ax.set_ylim(0.5, 2.5)
show(ax, "Rolling hedge ratio converging to the true beta")

## The spread and its drift

`RollingSpread(window_size)(x, y)` returns `x - beta * y` at each step, where beta is the same rolling estimate as `RollingBeta`. This residual removes the co-movement driven by the shared factor and leaves only the stationary part. `RollingAlpha(window_size)(x, y)` computes the companion intercept: the regression constant from the same window, which tracks any level shift in the spread over time. Together, spread and alpha separate the long-run drift from the mean-reverting residual.

In [ ]:
spread = RollingSpread(window_size=W)(a, b)
alpha  = RollingAlpha(window_size=W)(a, b)

# Compute the mean ignoring NaN warmup values
spread_mean = np.nanmean(spread)

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)

axes[0].plot(spread, lw=1.4, color="steelblue", label=f"RollingSpread(window_size={W})")
axes[0].axhline(spread_mean, color="0.45", ls="--", lw=1.1,
                label=f"spread mean = {spread_mean:.2f}")
axes[0].set_title("Beta-adjusted spread")
axes[0].legend(fontsize=8)

axes[1].plot(alpha, lw=1.4, color="darkorange", label=f"RollingAlpha(window_size={W})")
axes[1].axhline(a0, color="0.35", ls="--", lw=1.2,
                label=f"true intercept = {a0}")
axes[1].set_title("Rolling intercept (alpha)")
axes[1].set_xlabel("sample")
axes[1].legend(fontsize=8)

plt.tight_layout()

## Mean reversion with RollingOU

`RollingOU(window_size, output=...)` fits a discrete-time Ornstein-Uhlenbeck model to the spread in a rolling window and returns one of four parameters selected by a string mode: `"mrr"` for the mean-reversion rate (a value in (0, 1) means the spread pulls back toward the mean at each step), `"mean"` for the fitted long-run mean, `"relmean"` for the mean relative to the current spread value, and `"std"` for the noise standard deviation. Higher `mrr` values indicate faster reversion.

In [ ]:
OU_W = 120
ou_mrr  = RollingOU(window_size=OU_W, output="mrr")(spread)
ou_mean = RollingOU(window_size=OU_W, output="mean")(spread)
ou_std  = RollingOU(window_size=OU_W, output="std")(spread)

fig, axes = plt.subplots(3, 1, figsize=(9, 7.5), sharex=True)

# Panel 1: spread with OU fitted mean and 1-sigma band
axes[0].plot(spread, lw=1.2, color="steelblue", alpha=0.7, label="spread")
axes[0].plot(ou_mean, lw=1.8, color="darkorange", label="OU fitted mean")
axes[0].fill_between(
    range(N),
    ou_mean - ou_std,
    ou_mean + ou_std,
    color="darkorange", alpha=0.18,
    label="+/- 1 OU std"
)
axes[0].set_title("Spread with OU fitted mean and noise band")
axes[0].legend(fontsize=8)

# Panel 2: mean-reversion rate
axes[1].plot(ou_mrr, lw=1.6, color="seagreen", label='RollingOU(output="mrr")')
axes[1].axhline(0.0, color="0.6", lw=0.8, ls=":")
axes[1].axhline(1.0, color="0.6", lw=0.8, ls=":")
axes[1].set_title("Mean-reversion rate (mrr): 0 < mrr < 1 means reverting")
axes[1].legend(fontsize=8)

# Panel 3: noise standard deviation
axes[2].plot(ou_std, lw=1.6, color="tomato", label='RollingOU(output="std")')
axes[2].set_title("OU noise standard deviation")
axes[2].set_xlabel("sample")
axes[2].legend(fontsize=8)

plt.tight_layout()

The spread oscillates around the OU fitted mean, and the mean-reversion rate stays in (0, 1) throughout, confirming that the synthetic pair is genuinely cointegrated. The OU half-life in steps is approximately `-1 / log(1 - mrr)` once `mrr` has settled: a higher rate means the spread covers half the distance to its mean in fewer steps, giving a tighter and faster-trading signal.